<a href="https://colab.research.google.com/github/jagadambika13/code/blob/main/model_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# **Load and Read the Dataset**

In [ ]:
data= pd.read_csv("/content/creditcard.csv")
data.head()

# **Explore the Dataset**

In [ ]:
print("Shape of dataset:", data.shape)
print("\nColumn names:\n", data.columns)
print("\nData types:\n", data.dtypes)

In [ ]:
print("\nClass distribution:")
print(data["Class"].value_counts())

In [ ]:
print("\nClass distribution in percentage:")
print(data["Class"].value_counts(normalize=True) * 100)

In [ ]:
data.describe()

In [ ]:
print(data["Class"].isnull().sum())

In [ ]:
data = data.dropna(subset=["Class"])

In [ ]:
print(data["Class"].isnull().sum())

# **Define Features and Target**

In [ ]:
X = data.drop("Class", axis=1)
y = data["Class"]

# **Split the Data into Train and Test Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# **Create Pipeline for Logistic Regression**

In [ ]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# **Create Pipeline for Random Forest**

In [ ]:
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(random_state=42))
])

# **Train Baseline Logistic Regression Model**

In [ ]:
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

print("Logistic Regression Results")
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

In [ ]:
print(classification_report(y_test, y_pred_lr))

# **Train Baseline Random Forest Model**

In [ ]:
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print("Random Forest Results")
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

In [ ]:
print(classification_report(y_test, y_pred_rf))

# **Create 5-Fold Cross-Validation Strategy**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle= True, random_state=42)

# **Hyperparameter Tuning for Logistic Regression using GridSearchCV**

In [ ]:
param_grid_lr = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__solver": ["liblinear", "lbfgs"]
}

In [ ]:
grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)

print("Best Logistic Regression Parameters:", grid_lr.best_params_)
print("Best Logistic Regression CV Score:", grid_lr.best_score_)

# **Hyperparameter Tuning for Random Forest using GridSearchCV**

In [ ]:
param_grid_rf = {
    "classifier__n_estimators": [100],
    "classifier__max_depth": [5, 10]
}

In [ ]:
cv = 3
from sklearn.model_selection import GridSearchCV

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best Score:", grid_rf.best_score_)

# **Evaluate the Best Logistic Regression Model**

In [ ]:
best_lr = grid_lr.best_estimator_
y_pred_best_lr = best_lr.predict(X_test)

precision_lr = precision_score(y_test, y_pred_best_lr)
recall_lr = recall_score(y_test, y_pred_best_lr)
f1_lr = f1_score(y_test, y_pred_best_lr)

print("Optimized Logistic Regression")
print("Precision:", precision_lr)
print("Recall:", recall_lr)
print("F1 Score:", f1_lr)

In [ ]:
print(classification_report(y_test, y_pred_best_lr))

# **Evaluate the Best Random Forest Model**

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

precision_rf = precision_score(y_test, y_pred_best_rf)
recall_rf = recall_score(y_test, y_pred_best_rf)
f1_rf = f1_score(y_test, y_pred_best_rf)

print("Optimized Random Forest")
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1 Score:", f1_rf)

In [ ]:
print(classification_report(y_test, y_pred_best_rf))

# **Final Comparison Table**

In [ ]:
results_table = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [precision_lr, precision_rf],
    "Recall": [recall_lr, recall_rf],
    "F1 Score": [f1_lr, f1_rf]
})

results_table

# **Baseline Comparison Table**

In [ ]:
baseline_results = pd.DataFrame({
    "Model": ["Baseline Logistic Regression", "Baseline Random Forest"],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ]
})

baseline_results

# **Combine Baseline and Optimised Results**

In [ ]:
final_results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "Optimized Logistic Regression",
        "Baseline Random Forest",
        "Optimized Random Forest"
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_lr,
        precision_score(y_test, y_pred_rf),
        precision_rf
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_lr,
        recall_score(y_test, y_pred_rf),
        recall_rf
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_lr,
        f1_score(y_test, y_pred_rf),
        f1_rf
    ]
})

final_results

# **Confusion Matrix for Logistic Regression model**



In [ ]:
print("Confusion Matrix - Optimized Logistic Regression")
print(confusion_matrix(y_test, y_pred_best_lr))

# **Confusion Matrix for Random Forest Regression model**

In [ ]:
print("Confusion Matrix - Optimized Random Forest")
print(confusion_matrix(y_test, y_pred_best_rf))

# **Data Visualisation**

**Q1: What is the distribution of fraudulent vs non-fraudulent transactions?**

In [ ]:
import matplotlib.pyplot as plt
class_counts = data["Class"].value_counts()

labels = ["Non-Fraud (0)", "Fraud (1)"]
colors = ["#4CAF50", "#FF5252"]

plt.figure(figsize=(6,6))
plt.pie(class_counts, labels=labels, autopct='%1.2f%%', colors=colors, startangle=90, explode=(0,0.1))
plt.title("Distribution of Fraud vs Non-Fraud Transactions", fontsize=14)
plt.legend(loc="upper right")
plt.show()

**Q2: How many transactions are fraud vs non-fraud?**

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,4))
sns.countplot(x="Class", data=data, palette=["#2196F3", "#FF5722"])

plt.title("Transaction Count by Class", fontsize=14)
plt.xlabel("Transaction Type (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Number of Transactions")

for i, v in enumerate(data["Class"].value_counts()):
    plt.text(i, v + 500, str(v), ha='center', fontsize=10)

plt.show()

**Q3: How does transaction amount vary over time?**

In [ ]:
plt.figure(figsize=(10,5))

data_sorted = data.sort_values(by="Time")

plt.plot(data_sorted["Time"][:1000], data_sorted["Amount"][:1000], color="#673AB7")

plt.title("Transaction Amount over Time (Sample)", fontsize=14)
plt.xlabel("Time")
plt.ylabel("Transaction Amount")

plt.grid(True)
plt.show()